# Phase 7 — PySpark Performance Engineering Experiments Notebook

This is the **worked SOLUTION notebook** for Phase 7.

Run it **top-to-bottom**. Every experiment follows:

```text
baseline
    ↓
predict bottleneck
    ↓
inspect evidence
    ↓
execute / measure
    ↓
change ONE thing
    ↓
inspect again
    ↓
execute / measure again
    ↓
compare
    ↓
reconcile correctness
    ↓
explain WHY
```

Core rule:

> **Fix query and data design before tuning configuration.**

The notebook reuses Phase 4–6 concepts such as `Exchange`, `BroadcastHashJoin`,
`SortMergeJoin`, Parquet pruning, execution partitions, `repartition()`,
`coalesce()`, AQE, and skew without substantially reteaching them.

**Scope:** Phase 7 practice only. This notebook does not perform the formal
mastery gate, update `ROADMAP.md`, mark Phase 7 complete, or enter Phase 8
Spark UI diagnostics.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Experiment Protocol](#experiment-protocol)
- [Experiment 1 — Diagnose Unnecessary I/O](#experiment-1)
- [Experiment 2 — File Sizing: Fix the Writer](#experiment-2)
- [Experiment 3 — Broadcast vs. Shuffle Join](#experiment-3)
- [Experiment 4 — Remove an Unnecessary Shuffle](#experiment-4)
- [Experiment 5 — Diagnose Skew and Apply Salting](#experiment-5)
- [Experiment 6 — Justified vs. Unnecessary Caching](#experiment-6)
- [Experiment 7 — AQE Runtime Improvements](#experiment-7)
- [Applied Phase 7 Project](#applied-project)

---

<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Main grains:

```text
fact_sales_df
= one row per sale_id

dim_store_df
= one row per store_id

dim_product_df
= one row per product_id

skewed_sales_df
= one row per sale_id
```


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


spark = (
    SparkSession.builder
    .appName('phase_07_performance_experiments')
    .master('local[4]')
    # Keep ordinary experiments static. AQE is enabled only in Experiment 7.
    .config('spark.sql.shuffle.partitions', '12')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate()
)


In [ ]:
# Grain: one row per sale_id.
fact_sales_df = (
    spark.range(
        start=0,
        end=120000,
        step=1,
        numPartitions=8,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.date_add(
            F.lit('2026-01-01').cast('date'),
            (F.col('id') % 180).cast('int'),
        ).alias('order_date'),
        F.concat(
            F.lit('S'),
            F.lpad(
                ((F.col('id') % 40) + F.lit(1)).cast('string'),
                3,
                '0',
            ),
        ).alias('store_id'),
        F.concat(
            F.lit('P'),
            F.lpad(
                ((F.col('id') % 500) + F.lit(1)).cast('string'),
                4,
                '0',
            ),
        ).alias('product_id'),
        F.concat(
            F.lit('C'),
            F.lpad(
                ((F.col('id') % 8000) + F.lit(1)).cast('string'),
                5,
                '0',
            ),
        ).alias('customer_id'),
        F.when(
            (F.col('id') % 10) < 8,
            F.lit('COMPLETED'),
        )
        .when(
            (F.col('id') % 10) == 8,
            F.lit('CANCELLED'),
        )
        .otherwise(F.lit('RETURNED'))
        .alias('order_status'),
        ((F.col('id') % 5) + F.lit(1)).cast('int').alias('quantity'),
        (
            F.lit(5.00)
            + ((F.col('id') % 75) * F.lit(0.75))
        )
        .cast(DecimalType(12, 2))
        .alias('unit_price'),
        F.concat(
            F.lit('PROMO_'),
            (F.col('id') % 25).cast('string'),
        ).alias('promotion_code'),
        F.concat(
            F.lit('CHANNEL_'),
            (F.col('id') % 4).cast('string'),
        ).alias('sales_channel'),
        F.concat(
            F.lit('DEVICE_'),
            (F.col('id') % 6).cast('string'),
        ).alias('device_type'),
    )
    .withColumn('year', F.year('order_date'))
    .withColumn('month', F.month('order_date'))
    .withColumn(
        'gross_sales',
        (F.col('quantity') * F.col('unit_price')).cast(DecimalType(16, 2)),
    )
)


# Grain: one row per store_id.
dim_store_df = (
    spark.range(
        start=1,
        end=41,
        step=1,
        numPartitions=2,
    )
    .select(
        F.concat(
            F.lit('S'),
            F.lpad(F.col('id').cast('string'), 3, '0'),
        ).alias('store_id'),
        F.concat(
            F.lit('Store '),
            F.col('id').cast('string'),
        ).alias('store_name'),
        F.when(F.col('id') <= 20, F.lit('ON'))
        .otherwise(F.lit('BC'))
        .alias('province'),
        F.when((F.col('id') % 2) == 0, F.lit('URBAN'))
        .otherwise(F.lit('SUBURBAN'))
        .alias('store_format'),
    )
)


# Grain: one row per product_id.
dim_product_df = (
    spark.range(
        start=1,
        end=501,
        step=1,
        numPartitions=4,
    )
    .select(
        F.concat(
            F.lit('P'),
            F.lpad(F.col('id').cast('string'), 4, '0'),
        ).alias('product_id'),
        F.concat(
            F.lit('Product '),
            F.col('id').cast('string'),
        ).alias('product_name'),
        F.concat(
            F.lit('CATEGORY_'),
            (F.col('id') % 12).cast('string'),
        ).alias('category'),
        F.concat(
            F.lit('BRAND_'),
            (F.col('id') % 30).cast('string'),
        ).alias('brand'),
    )
)


# Grain: one row per sale_id.
# HOT_STORE deliberately owns 70% of the rows.
skewed_sales_df = (
    spark.range(
        start=0,
        end=100000,
        step=1,
        numPartitions=8,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.when(
            F.col('id') < 70000,
            F.lit('HOT_STORE'),
        )
        .otherwise(
            F.concat(
                F.lit('COLD_'),
                F.lpad(
                    ((F.col('id') % 30) + F.lit(1)).cast('string'),
                    2,
                    '0',
                ),
            )
        )
        .alias('store_id'),
        ((F.col('id') % 5) + F.lit(1)).cast('int').alias('quantity'),
        (
            F.lit(10.00)
            + ((F.col('id') % 25) * F.lit(0.50))
        )
        .cast(DecimalType(12, 2))
        .alias('unit_price'),
    )
    .withColumn(
        'gross_sales',
        (F.col('quantity') * F.col('unit_price')).cast(DecimalType(16, 2)),
    )
)


In [ ]:
def show_partition_summary(df, label):
    '''Print execution-partition count and non-empty row distribution.'''

    rows_by_partition = (
        df
        .select(F.spark_partition_id().alias('partition_id'))
        .groupBy('partition_id')
        .agg(F.count('*').alias('row_count'))
        .orderBy('partition_id')
        .collect()
    )

    print(f'\n{label}')
    print('execution partitions:', df.rdd.getNumPartitions())

    for row in rows_by_partition:
        print(
            f'partition {row.partition_id}: '
            f'{row.row_count} rows'
        )


def time_count(df, label):
    '''Materialize the same count workload and report elapsed local time.'''

    started_at = perf_counter()
    row_count = df.count()
    elapsed_seconds = perf_counter() - started_at

    print(f'{label}: {row_count} rows in {elapsed_seconds:.3f} s')
    return row_count, elapsed_seconds


def parquet_data_files(path):
    '''Return only physical Parquet data files below a local path.'''

    return sorted(Path(path).rglob('*.parquet'))


def show_file_summary(path, label):
    '''Print compact physical Parquet file-size evidence.'''

    files = parquet_data_files(path)
    sizes = [file_path.stat().st_size for file_path in files]

    print(f'\n{label}')
    print('parquet data files:', len(files))

    if sizes:
        print('minimum bytes:', min(sizes))
        print('maximum bytes:', max(sizes))
        print('average bytes:', round(sum(sizes) / len(sizes), 2))
        print('total bytes:', sum(sizes))


def scalar_sum(df, column_name):
    '''Return one deterministic numeric reconciliation value.'''

    return (
        df
        .agg(F.sum(column_name).alias('value'))
        .first()
        .value
    )


def assert_same_rows(df_a, df_b, columns):
    '''Assert exact multiset equality for deterministic teaching outputs.'''

    left = df_a.select(*columns)
    right = df_b.select(*columns)

    assert left.exceptAll(right).count() == 0
    assert right.exceptAll(left).count() == 0


In [ ]:
# Keep this object alive for the entire notebook session.
temp_directory = TemporaryDirectory(prefix='pyspark_phase_07_')
temp = Path(temp_directory.name)

partitioned_sales_path = temp / 'fact_sales_partitioned'

(
    fact_sales_df
    .write
    .mode('overwrite')
    .partitionBy('year', 'month')
    .parquet(str(partitioned_sales_path))
)

print('temporary root:', temp)
print('fact rows:', fact_sales_df.count())
print('fact execution partitions:', fact_sales_df.rdd.getNumPartitions())


[Back to Table of Contents](#toc)

---

<a id="experiment-protocol"></a>
# Experiment Protocol

For every experiment, keep the business result fixed and change exactly one
performance variable.

```text
Business requirement
Input grain(s)
Output grain
Correctness invariants
Suspected bottleneck
Evidence
Materializing action
```

Then compare physical behavior, runtime support, and correctness.


[Back to Table of Contents](#toc)

---

<a id="experiment-1"></a>
# Experiment 1 — Diagnose Unnecessary I/O

**Requirement:** completed June 2026 sales by store.

**Output grain:** one row per `store_id`.

The partitioned Parquet dataset is physically organized by `year` and `month`.
The baseline filters June using only `order_date`, which is logically correct but
does not directly constrain the stored partition columns.

### Prediction

- Required business columns: `store_id`, `quantity`, `gross_sales`.
- `order_status` is needed for row filtering.
- The baseline date predicate may appear in `PushedFilters`.
- The baseline should lack direct `year` / `month` `PartitionFilters`.
- Equivalent `year` / `month` predicates should enable partition pruning.


In [ ]:
baseline_io_df = (
    spark.read.parquet(str(partitioned_sales_path))
    .filter(
        (F.col('order_date') >= F.lit('2026-06-01').cast('date'))
        & (F.col('order_date') < F.lit('2026-07-01').cast('date'))
        & (F.col('order_status') == 'COMPLETED')
    )
    .select(
        'store_id',
        'quantity',
        'gross_sales',
    )
    .groupBy('store_id')
    .agg(
        F.sum('quantity').alias('units'),
        F.sum('gross_sales').alias('gross_sales'),
    )
)

print('BASELINE I/O PLAN')
baseline_io_df.explain('formatted')

baseline_io_count, baseline_io_seconds = time_count(
    baseline_io_df,
    'BASELINE I/O ACTION',
)


### Baseline diagnosis

The strongest first bottleneck is unnecessary physical history scanning: the
business request is one month, but the baseline predicate does not directly
constrain the `year` / `month` storage partition columns.

Column pruning is already something Spark can perform through the optimized scan,
so inspect `ReadSchema` rather than assuming that moving `select()` earlier must
help.


In [ ]:
optimized_io_df = (
    spark.read.parquet(str(partitioned_sales_path))
    # Change ONE thing: express June through the stored partition columns.
    .filter(
        (F.col('year') == 2026)
        & (F.col('month') == 6)
        & (F.col('order_status') == 'COMPLETED')
    )
    .select(
        'store_id',
        'quantity',
        'gross_sales',
    )
    .groupBy('store_id')
    .agg(
        F.sum('quantity').alias('units'),
        F.sum('gross_sales').alias('gross_sales'),
    )
)

print('OPTIMIZED I/O PLAN')
optimized_io_df.explain('formatted')

optimized_io_count, optimized_io_seconds = time_count(
    optimized_io_df,
    'OPTIMIZED I/O ACTION',
)


In [ ]:
assert baseline_io_count == optimized_io_count
assert_same_rows(
    baseline_io_df,
    optimized_io_df,
    ['store_id', 'units', 'gross_sales'],
)

print('baseline units:', scalar_sum(baseline_io_df, 'units'))
print('optimized units:', scalar_sum(optimized_io_df, 'units'))
print('baseline gross sales:', scalar_sum(baseline_io_df, 'gross_sales'))
print('optimized gross sales:', scalar_sum(optimized_io_df, 'gross_sales'))


### Worked explanation

**Bottleneck:** the June-only workload was not expressed in the stored partition
vocabulary.

**Evidence:** compare `PartitionFilters` in the two scans.

**Change:** replace the date-range predicate with equivalent `year` / `month`
storage-partition predicates.

**Why:** Spark can reject irrelevant storage directories before scanning their
Parquet data.

**Correctness:** same store-level rows, units, and gross-sales totals.


[Back to Table of Contents](#toc)

---

<a id="experiment-2"></a>
# Experiment 2 — File Sizing: Fix the Writer

The same logical fact data is written twice.

```text
baseline: 96 output execution partitions
one change: 8 output execution partitions
```

No schema, codec, storage partitioning, writer file-cap, or read setting changes.


In [ ]:
fragmented_path = temp / 'fragmented_fact'
improved_file_path = temp / 'improved_fact'

fragmented_source_df = fact_sales_df.repartition(96)

print('fragmented writer partitions:', fragmented_source_df.rdd.getNumPartitions())

(
    fragmented_source_df
    .write
    .mode('overwrite')
    .parquet(str(fragmented_path))
)

show_file_summary(
    fragmented_path,
    'FRAGMENTED BASELINE FILE LAYOUT',
)

fragmented_read_df = spark.read.parquet(str(fragmented_path))

print(
    'fragmented next-read execution partitions:',
    fragmented_read_df.rdd.getNumPartitions(),
)

fragmented_count, fragmented_seconds = time_count(
    fragmented_read_df,
    'FRAGMENTED RE-READ',
)


In [ ]:
# Change ONE thing: writer execution partition count.
improved_file_source_df = fact_sales_df.repartition(8)

print(
    'improved writer partitions:',
    improved_file_source_df.rdd.getNumPartitions(),
)

(
    improved_file_source_df
    .write
    .mode('overwrite')
    .parquet(str(improved_file_path))
)

show_file_summary(
    improved_file_path,
    'IMPROVED FILE LAYOUT',
)

improved_file_read_df = spark.read.parquet(str(improved_file_path))

print(
    'improved next-read execution partitions:',
    improved_file_read_df.rdd.getNumPartitions(),
)

improved_file_count, improved_file_seconds = time_count(
    improved_file_read_df,
    'IMPROVED RE-READ',
)


In [ ]:
assert fragmented_count == improved_file_count == fact_sales_df.count()
assert fragmented_read_df.schema == improved_file_read_df.schema
assert (
    scalar_sum(fragmented_read_df, 'gross_sales')
    == scalar_sum(improved_file_read_df, 'gross_sales')
)

print('fragmented files:', len(parquet_data_files(fragmented_path)))
print('improved files:', len(parquet_data_files(improved_file_path)))


### Worked explanation

Excessive tiny files can add listing, open, scan-planning, and task overhead.
Reducing writer parallelism creates fewer, larger files for this teaching-sized
dataset.

But `fewer files != always better`: over-compaction can reduce useful parallelism
and create oversized tasks. Fix a recurring file-layout pathology at the writer
rather than permanently compensating for it with arbitrary reader tuning.


[Back to Table of Contents](#toc)

---

<a id="experiment-3"></a>
# Experiment 3 — Broadcast vs. Shuffle Join

**Requirement:** enrich every sale with store attributes.

```text
fact_sales_df = many rows per store_id
dim_store_df = one row per store_id
output grain = one row per sale_id
```

Validate the dimension grain before judging join performance.


In [ ]:
dim_store_rows = dim_store_df.count()
dim_store_unique_keys = dim_store_df.select('store_id').distinct().count()

assert dim_store_rows == dim_store_unique_keys

print('dimension rows:', dim_store_rows)
print('dimension unique store_ids:', dim_store_unique_keys)


In [ ]:
original_broadcast_threshold = spark.conf.get(
    'spark.sql.autoBroadcastJoinThreshold'
)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

baseline_join_df = (
    fact_sales_df
    .select(
        'sale_id',
        'store_id',
        'gross_sales',
    )
    .hint('merge')
    .join(
        dim_store_df
        .select(
            'store_id',
            'store_name',
            'province',
            'store_format',
        )
        .hint('merge'),
        on='store_id',
        how='left',
    )
)

print('BASELINE SHUFFLE JOIN PLAN')
baseline_join_df.explain('formatted')

baseline_join_count, baseline_join_seconds = time_count(
    baseline_join_df,
    'BASELINE SHUFFLE JOIN',
)


In [ ]:
# Change ONE thing: broadcast the proven-small, unique dimension.
optimized_join_df = (
    fact_sales_df
    .select(
        'sale_id',
        'store_id',
        'gross_sales',
    )
    .join(
        F.broadcast(
            dim_store_df.select(
                'store_id',
                'store_name',
                'province',
                'store_format',
            )
        ),
        on='store_id',
        how='left',
    )
)

print('OPTIMIZED BROADCAST JOIN PLAN')
optimized_join_df.explain('formatted')

optimized_join_count, optimized_join_seconds = time_count(
    optimized_join_df,
    'OPTIMIZED BROADCAST JOIN',
)


In [ ]:
assert baseline_join_count == optimized_join_count == fact_sales_df.count()
assert baseline_join_df.select('sale_id').distinct().count() == baseline_join_count
assert optimized_join_df.select('sale_id').distinct().count() == optimized_join_count
assert (
    scalar_sum(baseline_join_df, 'gross_sales')
    == scalar_sum(optimized_join_df, 'gross_sales')
)

spark.conf.set(
    'spark.sql.autoBroadcastJoinThreshold',
    original_broadcast_threshold,
)


### Worked explanation

A sort-merge path commonly requires compatible join-key distribution on both
sides plus sorting. A broadcast join replicates the genuinely small build side,
allowing the large fact partitions to probe it without ordinary fact-side
join-key redistribution.

A table being called a dimension is not enough: actual size, projected width,
memory safety, and key uniqueness still matter.


[Back to Table of Contents](#toc)

---

<a id="experiment-4"></a>
# Experiment 4 — Remove an Unnecessary Shuffle

**Requirement:** completed sales totals by `product_id`.

Baseline smell:

```text
repartition(12, 'store_id')
→ groupBy('product_id')
```

The first distribution does not satisfy the second wide requirement.


In [ ]:
baseline_shuffle_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .repartition(12, 'store_id')
    .groupBy('product_id')
    .agg(
        F.sum('gross_sales').alias('gross_sales'),
        F.sum('quantity').alias('units'),
    )
)

print('BASELINE REDUNDANT-SHUFFLE PLAN')
baseline_shuffle_df.explain('formatted')

baseline_shuffle_count, baseline_shuffle_seconds = time_count(
    baseline_shuffle_df,
    'BASELINE REDUNDANT SHUFFLE',
)


In [ ]:
# Change ONE thing: remove only the store_id repartition.
optimized_shuffle_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .groupBy('product_id')
    .agg(
        F.sum('gross_sales').alias('gross_sales'),
        F.sum('quantity').alias('units'),
    )
)

print('OPTIMIZED SHUFFLE PLAN')
optimized_shuffle_df.explain('formatted')

optimized_shuffle_count, optimized_shuffle_seconds = time_count(
    optimized_shuffle_df,
    'OPTIMIZED NECESSARY SHUFFLE',
)


In [ ]:
assert baseline_shuffle_count == optimized_shuffle_count
assert_same_rows(
    baseline_shuffle_df,
    optimized_shuffle_df,
    ['product_id', 'gross_sales', 'units'],
)

print(
    'baseline output partitions:',
    baseline_shuffle_df.rdd.getNumPartitions(),
)
print(
    'optimized output partitions:',
    optimized_shuffle_df.rdd.getNumPartitions(),
)


### Worked explanation

The optimized pipeline still shuffles because `groupBy('product_id')` requires
equal product states to meet.

The removed shuffle was the earlier `store_id` redistribution. It had no
downstream value because the next wide requirement uses `product_id`.


[Back to Table of Contents](#toc)

---

<a id="experiment-5"></a>
# Experiment 5 — Diagnose Skew and Apply Targeted Salting

`HOT_STORE` deliberately owns 70% of the data.

The first question is not merely “how many partitions?” but “how are rows
distributed across the key space?”


In [ ]:
key_frequency_df = (
    skewed_sales_df
    .groupBy('store_id')
    .agg(F.count('*').alias('row_count'))
    .orderBy(
        F.col('row_count').desc(),
        F.col('store_id').asc(),
    )
)

key_frequency_df.show(10, truncate=False)

total_skew_rows = skewed_sales_df.count()
hot_rows = (
    key_frequency_df
    .filter(F.col('store_id') == 'HOT_STORE')
    .first()
    .row_count
)

print('HOT_STORE share:', round(hot_rows / total_skew_rows, 4))


In [ ]:
skewed_by_key_df = skewed_sales_df.repartition(
    12,
    'store_id',
)

show_partition_summary(
    skewed_by_key_df,
    'HASH PARTITIONED BY SKEWED STORE_ID',
)


### Diagnosis

Twelve partitions can still be badly distributed. Normal hash repartitioning
keeps equal `store_id` values together, so changing only `12 → 100` buckets still
does not split one equal hot key across many partitions.


In [ ]:
unsalted_store_sales_df = (
    skewed_sales_df
    .groupBy('store_id')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

unsalted_store_sales_df.orderBy('store_id').show(40, truncate=False)


In [ ]:
# Change ONE thing: add a deterministic salt that varies inside the hot key.
salted_input_df = skewed_sales_df.withColumn(
    'salt',
    F.pmod(
        F.xxhash64('sale_id'),
        F.lit(8),
    ),
)

# Stage 1 grain = one row per (store_id, salt).
salted_partial_df = (
    salted_input_df
    .groupBy(
        'store_id',
        'salt',
    )
    .agg(
        F.sum('gross_sales').alias('partial_gross_sales')
    )
)

# Stage 2 removes the physical salt and restores business grain.
salted_store_sales_df = (
    salted_partial_df
    .groupBy('store_id')
    .agg(
        F.sum('partial_gross_sales').alias('gross_sales')
    )
)

print('SALTED TWO-STAGE PLAN')
salted_store_sales_df.explain('formatted')


In [ ]:
hot_salt_distribution_df = (
    salted_input_df
    .filter(F.col('store_id') == 'HOT_STORE')
    .groupBy('salt')
    .agg(F.count('*').alias('row_count'))
    .orderBy('salt')
)

hot_salt_distribution_df.show(truncate=False)

assert_same_rows(
    unsalted_store_sales_df,
    salted_store_sales_df,
    ['store_id', 'gross_sales'],
)

print(
    'unsalted total:',
    scalar_sum(unsalted_store_sales_df, 'gross_sales'),
)
print(
    'salted total:',
    scalar_sum(salted_store_sales_df, 'gross_sales'),
)


### Worked explanation

The problem is **distribution**, not just partition count.

The salt must vary within `HOT_STORE`; hashing only `store_id` would assign every
hot row the same salt. Salt is a physical distribution aid, not part of the
business key. The second aggregation restores one row per `store_id`.

Manual salting should follow simpler checks: invalid/sentinel hot keys, earlier
volume reduction, broadcast opportunities, and AQE skew handling.


[Back to Table of Contents](#toc)

---

<a id="experiment-6"></a>
# Experiment 6 — Justified vs. Unnecessary Caching

Cache only when the same expensive lineage is reused enough to justify its
storage cost.


In [ ]:
reused_sales_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .select(
        'sale_id',
        'store_id',
        'product_id',
        'gross_sales',
    )
    .join(
        F.broadcast(
            dim_store_df.select(
                'store_id',
                'province',
                'store_format',
            )
        ),
        on='store_id',
        how='left',
    )
    .withColumn(
        'sales_band',
        F.when(F.col('gross_sales') >= F.lit(100), F.lit('HIGH'))
        .otherwise(F.lit('STANDARD')),
    )
)

province_workload_df = (
    reused_sales_df
    .groupBy('province')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

format_workload_df = (
    reused_sales_df
    .groupBy('store_format')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

product_workload_df = (
    reused_sales_df
    .groupBy('product_id')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)


In [ ]:
def run_three_reuse_actions(label):
    started_at = perf_counter()

    province_rows = province_workload_df.count()
    format_rows = format_workload_df.count()
    product_rows = product_workload_df.count()

    elapsed_seconds = perf_counter() - started_at

    print(
        f'{label}: '
        f'province_rows={province_rows}, '
        f'format_rows={format_rows}, '
        f'product_rows={product_rows}, '
        f'elapsed={elapsed_seconds:.3f}s'
    )

    return elapsed_seconds


uncached_reuse_seconds = run_three_reuse_actions(
    'UNCACHED THREE-ACTION SEQUENCE'
)


In [ ]:
# Change ONE thing: persist the reduced reusable intermediate.
reused_sales_df.persist(StorageLevel.MEMORY_AND_DISK)

print('storage level:', reused_sales_df.storageLevel)

cache_materialization_count, cache_materialization_seconds = time_count(
    reused_sales_df,
    'CACHE MATERIALIZATION',
)

cached_reuse_seconds = run_three_reuse_actions(
    'CACHED THREE-ACTION REUSE SEQUENCE'
)

print('CACHED DOWNSTREAM PLAN')
province_workload_df.explain('formatted')


In [ ]:
assert cache_materialization_count == reused_sales_df.count()

reused_sales_df.unpersist()

print('storage level after unpersist:', reused_sales_df.storageLevel)


### When not to cache

```text
used once
→ no recomputation to avoid
→ cache population adds work/storage pressure
```

Prefer caching a reduced reusable intermediate over a raw wide fact table when
both serve the same reuse window. Cache pressure can cause eviction, disk fallback,
GC, and less memory for shuffle/sort/aggregation work.

Separate cold, cache-materializing, and cache-reusing timings. Unpersist after the
reuse window.


[Back to Table of Contents](#toc)

---

<a id="experiment-7"></a>
# Experiment 7 — AQE Runtime Improvements

AQE is evaluated after the static query/data design is reasonable.

Inspect what Spark actually does. Runtime join conversion and skew splitting may
or may not trigger on a local teaching workload.


In [ ]:
original_shuffle_partitions = spark.conf.get('spark.sql.shuffle.partitions')
original_aqe_enabled = spark.conf.get('spark.sql.adaptive.enabled')
original_auto_broadcast = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')

spark.conf.set('spark.sql.shuffle.partitions', '64')
spark.conf.set('spark.sql.adaptive.enabled', 'false')

static_aqe_baseline_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .groupBy('store_id')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

print('STATIC BASELINE PLAN')
static_aqe_baseline_df.explain('formatted')

static_aqe_count, static_aqe_seconds = time_count(
    static_aqe_baseline_df,
    'STATIC 64-PARTITION SHUFFLE',
)

print(
    'static result execution partitions:',
    static_aqe_baseline_df.rdd.getNumPartitions(),
)


In [ ]:
# Change ONE thing: enable AQE and rebuild the query.
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')

adaptive_grouped_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .groupBy('store_id')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

adaptive_count, adaptive_seconds = time_count(
    adaptive_grouped_df,
    'AQE GROUPED ACTION',
)

print(
    'adaptive result execution partitions:',
    adaptive_grouped_df.rdd.getNumPartitions(),
)

print('ADAPTIVE FINAL PLAN')
adaptive_grouped_df.explain('formatted')

assert static_aqe_count == adaptive_count
assert_same_rows(
    static_aqe_baseline_df,
    adaptive_grouped_df,
    ['store_id', 'gross_sales'],
)


### Part A — partition coalescing

With AQE enabled:

```text
planned shuffle partitions
!=
necessarily final runtime partitions
```

Record the final partition count and adaptive plan you actually observe.


In [ ]:
# Part B: allow adaptive statistics to consider runtime broadcast conversion.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
spark.conf.set(
    'spark.sql.adaptive.autoBroadcastJoinThreshold',
    '10485760',
)

aqe_join_df = (
    fact_sales_df
    .select(
        'sale_id',
        'store_id',
        'gross_sales',
    )
    .join(
        dim_store_df.select(
            'store_id',
            'province',
        ),
        on='store_id',
        how='inner',
    )
)

time_count(
    aqe_join_df,
    'AQE JOIN ACTION',
)

print('AQE JOIN INITIAL/FINAL PLAN')
aqe_join_df.explain('formatted')


### Part B — runtime join changes

If the final plan switches to broadcast, document the initial and final join
operators and the runtime evidence that made the build side small.

If it does not switch, record that fact instead of fabricating an adaptive
conversion.


In [ ]:
# Part C: inspect AQE skew handling without assuming it must trigger locally.
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set(
    'spark.sql.adaptive.autoBroadcastJoinThreshold',
    '-1',
)

skewed_store_dimension_df = (
    skewed_sales_df
    .select('store_id')
    .distinct()
    .withColumn('region', F.lit('REGION_1'))
)

aqe_skew_join_df = (
    skewed_sales_df
    .hint('merge')
    .join(
        skewed_store_dimension_df.hint('merge'),
        on='store_id',
        how='inner',
    )
    .groupBy('region')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

time_count(
    aqe_skew_join_df,
    'AQE SKEW JOIN ACTION',
)

print('AQE SKEW FINAL PLAN')
aqe_skew_join_df.explain('formatted')


### Part C — skew handling

Key-frequency evidence already proves skew exists. The final adaptive plan tells
you whether the local shuffle sizes crossed the runtime thresholds for AQE skew
handling.

AQE complements correct query/data design; it does not replace it.


In [ ]:
spark.conf.set(
    'spark.sql.autoBroadcastJoinThreshold',
    original_auto_broadcast,
)
spark.conf.set(
    'spark.sql.shuffle.partitions',
    original_shuffle_partitions,
)
spark.conf.set(
    'spark.sql.adaptive.enabled',
    original_aqe_enabled,
)

print(
    'restored shuffle partitions:',
    spark.conf.get('spark.sql.shuffle.partitions'),
)
print(
    'restored AQE enabled:',
    spark.conf.get('spark.sql.adaptive.enabled'),
)


[Back to Table of Contents](#toc)

---

<a id="applied-project"></a>
# Applied Phase 7 Project — Diagnose and Optimize One Pipeline

**Business requirement:** June 2026 completed sales by `province` and `category`
with `units` and `gross_sales`.

Expected output grain:

```text
one row per (province, category)
```

The baseline is logically correct but intentionally inefficient. The worked
solution chooses one first optimization and leaves the other candidates intact.


In [ ]:
assert (
    dim_store_df.select('store_id').distinct().count()
    == dim_store_df.count()
)
assert (
    dim_product_df.select('product_id').distinct().count()
    == dim_product_df.count()
)

print('fact grain: one row per sale_id')
print('store dimension grain: one row per store_id')
print('product dimension grain: one row per product_id')
print('output grain: one row per (province, category)')


In [ ]:
original_applied_broadcast_threshold = spark.conf.get(
    'spark.sql.autoBroadcastJoinThreshold'
)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

applied_baseline_df = (
    spark.read.parquet(str(partitioned_sales_path))
    # Smell 1: correct June filter, but not direct storage partition pruning.
    .filter(
        (F.col('order_date') >= F.lit('2026-06-01').cast('date'))
        & (F.col('order_date') < F.lit('2026-07-01').cast('date'))
        & (F.col('order_status') == 'COMPLETED')
    )
    # Smell 2: this store distribution has limited value for later work.
    .repartition(12, 'store_id')
    .join(
        dim_store_df
        .select(
            'store_id',
            'province',
        )
        .hint('merge'),
        on='store_id',
        how='inner',
    )
    .join(
        dim_product_df
        .select(
            'product_id',
            'category',
        )
        .hint('merge'),
        on='product_id',
        how='inner',
    )
    .groupBy(
        'province',
        'category',
    )
    .agg(
        F.sum('quantity').alias('units'),
        F.sum('gross_sales').alias('gross_sales'),
    )
)

print('APPLIED BASELINE PLAN')
applied_baseline_df.explain('formatted')

applied_baseline_count, applied_baseline_seconds = time_count(
    applied_baseline_df,
    'APPLIED BASELINE ACTION',
)

applied_baseline_units = scalar_sum(
    applied_baseline_df,
    'units',
)
applied_baseline_sales = scalar_sum(
    applied_baseline_df,
    'gross_sales',
)


## Worked diagnosis

**Candidate 1:** missing direct storage partition pruning.  
**Evidence:** June-only requirement + source partitioned by `year/month` +
baseline date-only filter.  
**Estimated importance:** high.

**Candidate 2:** forced shuffle joins against small, unique dimensions.  
**Estimated importance:** high.

**Candidate 3:** explicit repartition by `store_id` before later product/final
aggregation distributions.  
**Estimated importance:** medium/high.

### First optimization

Choose **partition pruning first** because it removes irrelevant physical input
before that data can reach any downstream join or shuffle.

For causal isolation, the forced merge joins, explicit repartition, and final
aggregation shuffle remain unchanged.


In [ ]:
applied_optimized_df = (
    spark.read.parquet(str(partitioned_sales_path))
    # Change ONE thing: equivalent June requirement through storage partitions.
    .filter(
        (F.col('year') == 2026)
        & (F.col('month') == 6)
        & (F.col('order_status') == 'COMPLETED')
    )
    .repartition(12, 'store_id')
    .join(
        dim_store_df
        .select(
            'store_id',
            'province',
        )
        .hint('merge'),
        on='store_id',
        how='inner',
    )
    .join(
        dim_product_df
        .select(
            'product_id',
            'category',
        )
        .hint('merge'),
        on='product_id',
        how='inner',
    )
    .groupBy(
        'province',
        'category',
    )
    .agg(
        F.sum('quantity').alias('units'),
        F.sum('gross_sales').alias('gross_sales'),
    )
)

print('APPLIED OPTIMIZED PLAN')
applied_optimized_df.explain('formatted')

applied_optimized_count, applied_optimized_seconds = time_count(
    applied_optimized_df,
    'APPLIED AFTER FIRST OPTIMIZATION',
)


In [ ]:
assert applied_baseline_count == applied_optimized_count
assert_same_rows(
    applied_baseline_df,
    applied_optimized_df,
    ['province', 'category', 'units', 'gross_sales'],
)

assert scalar_sum(applied_optimized_df, 'units') == applied_baseline_units
assert scalar_sum(
    applied_optimized_df,
    'gross_sales',
) == applied_baseline_sales

spark.conf.set(
    'spark.sql.autoBroadcastJoinThreshold',
    original_applied_broadcast_threshold,
)

print('APPLIED CORRECTNESS RECONCILED')


## Final diagnosis

**Bottleneck:** June-only work did not directly exploit `year/month` storage
partitioning.

**Evidence:** compare scan `PartitionFilters`.

**First optimization:** equivalent `year = 2026` and `month = 6` predicates.

**Before → after:** unrelated storage directories remain candidates before;
the optimized scan can prune them.

**Runtime:** use the actual before/after local timings as supporting evidence only.

**Correctness:** identical `(province, category)` rows, row count, units, and
gross-sales totals.

**Next:** investigate the forced shuffle joins, then the explicit `store_id`
repartition — one change at a time.


[Back to Table of Contents](#toc)

---

# Phase 7 Practice Complete — Mastery Gate Not Yet Performed

The notebook practiced:

```text
pruning
file sizing
broadcast vs. shuffle joins
unnecessary shuffle removal
skew / hot keys
salting
caching / persistence
AQE
correctness reconciliation
one-change-at-a-time diagnosis
```

Resource concepts remain downstream of query/data diagnosis. When memory pressure
appears, first identify whether the cause is an oversized/skewed partition, large
broadcast, cache pressure, sort/aggregation state, unnecessary shuffle volume, or
driver-side collection. Only then should executor/driver sizing become the
primary lever.
